# The union that swaps two columns

**Track:** debug-the-ai | **Difficulty:** easy | **Tags:** union, unionByName, schema, aggregation

Run the three setup cells below once, in order. Then read the problem, write your
answer in the answer cell so that it ends with a DataFrame called `result`, and run
`check(result)`. You can run the check as many times as you like.

Free Colab runs only two or three notebooks at a time. When you are done here, end the
session with Runtime > Disconnect and delete runtime before opening the next problem.

In [ ]:
# Setup: installs PySpark on Colab and starts a local Spark session (about a minute).
import os
import re
import shutil
import subprocess
import sys

PYSPARK_VERSION = "4.1.1"

if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", f"pyspark=={PYSPARK_VERSION}"],
        check=True,
    )
    # PySpark 4 needs Java 17 or newer.
    java_major = 0
    if shutil.which("java"):
        version_text = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
        match = re.search(r'version "(\d+)', version_text)
        java_major = int(match.group(1)) if match else 0
    if java_major < 17:
        subprocess.run(["apt-get", "install", "-y", "-qq", "openjdk-17-jdk-headless"], check=True)
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder.master("local[2]")
    .appName("008-union-swapped-columns")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "is ready.")

In [ ]:
#@title Load the data (run this cell, no need to read it)
import base64
import pathlib

_DATA = {
    "sales_north": """
UEFSMRUEFaALFe4FTBW0ARUAEgAA0AUMsa0BAAEBALINCACzDQgAtA0IALUNCAC2DQgAtw0IALgNCAC5DQgAug0IALsNCAC8DQgA
vQ0IAL4NCAC/DQgAwA0IAMENCADCDQgAww0IAMQNCADFDQgAxg0IAMcNCADIDQgAyQ0IAMoNCADLDQgAzA0IAM0NCADODQgAzw0I
ANANCADRDQgA0g0IANMNCADUDQgA1Q0IANYNCADXDQgA2A0IANkNCADaDQgA2w0IANwNCADdDQgA3g0IAN8NCADgDQgA4Q0IAOIN
CADjDQgA5A0IAOUNCADmDQgA5w0IAOgNCADpDQgA6g0IAOsNCADsDQgA7Q0IAO4NCADvDQgA8A0IAPENCADyDQgA8w0IAPQNCAD1
DQgA9g0IAPcNCAD4DQgA+Q0IAPoNCAD7DQgA/A0IAP0NCAD+DQgA/w0IBACuSXgAAQ0IAAINCAADDQgABA0IAAUNCAAGDQgABw0I
AAgNCDwJrgEAAAAAAAquAQAAAAAAFQAVugEVwAEsFbQBFRAVBhUGHBgICq4BAAAAAAAYCLGtAQAAAAAAFgAoCAquAQAAAAAAGAix
rQEAAAAAABERAAAAXfBcAwAAALQBAQcZgIBgQCgYDoiEYsFoOB6QiGRCqVgumIxmw+l4PqCQaEQqmU6olGrFarlesJhsRqvZbric
bsfr+X7AoHBILBqPyKRyyWw6n9CodEqtWq/YLAAAAAAAFQQVygEVzAFMFQ4VABIAAGXwPggAAABEZXNrIE1hdAkAAABVU0ItQyBI
dWILAAAATW9uaXRvciBBcm0GAAAAV2ViY2FtDQAAAFN0YW5kaW5nIAE7LAwAAABMYXB0b3AgUwEYRA4AAABXaXJlbGVzcyBNb3Vz
ZRUAFVoVXiwVtAEVEBUGFQYcNgAoDldpcmVsZXNzIE1vdXNlGAhEZXNrIE1hdBERAAAALbADAAAAtAEBAxkIlG1UymoBHReC40SB
UkwD23Wm7YlwAnXtNFUoE5WUSsAwAAAVBBWEBxWOB0wVtAEVABIAAMID9DQBAAAQcPwAAAdU9AAAEt4UAAAWwpAAAAE/7AAABkV4
AAAFbRAAAArxkAAAB2jgAAAUmdQAAAj2JAAAE4sgAAAJ9AwAAAkN+AAAErK4AAACbRgAABaXmAAAEhHIAAAGJwAAAAZGpAAAC8pc
AAAJtfAAABIANAAAD688AAAGkzQAAAc0JAAADmFAAAAVhvAAABUALAAACnv8AAAOonwAABDXVAAABAbcAAATFGAAAA8R0AAACYE0
AAANMbwAABX7vAAABHSgAAACCwwAAA2vIAAAFfnIAAADUZwAAAcgOAAAB8rsAAASgeQAABOA+AAAFjfkAAAPrhAAABOEfAAADbbw
AAAMq8AAABZsPAAABnE4AAACbLQAABTbEAAADcXIAAAMVpgAAA2lwAAADe/4AAASR0wAABCzAWTwiHsEAAAI6HgAAAY1dAAAEibg
AAABPZQAAAZn2AAABc/kAAAO3EwAABSkxAAAEO38AAAMIXgAABDK1AAACoVcAAADMAQAAA3c1AAAB2s4AAAUp+QAAAG4PAAAANLw
AAAGXUwAAAx9RAAAAevMAAAKNzwAABQVaAAAErfMAAASanQAAAXlxAAADgK4FQAVugEVwAEsFbQBFRAVBhUGHBgFAAAWwpAYBQAA
ANLwFgAoBQAAFsKQGAUAAADS8BERAAAAXfBcAwAAALQBAQcZgIBgQCgYDoiEYsFoOB6QiGRCqVgumIxmw+l4PqCQaEQqmU6olGrF
arlesJhsRqvZbricbsfr+X7AoHBILBqPyKRyyWw6n9CodEqtWq/YLAAAAAAAFQQVsAQVtARMFXAVABIAAJgCAAAJAfQQAQU8HAAA
AFBGAAABdwwAAABe2AAAArdkAAABfvAAAACbRgAABTI6AAAAbkAAAAERIQAAAeLwAAAChlgAAASADQAAA3NaAAABzQkAAAFKqAAA
AjH5AAAENdUAAAJ6+AAAA3dMAAAD04sAAADNUAAAAH2IAAAFtrQAAAC66gAAARGiAAABewkAAAJn7QAABY35AAADSqAAAAa6EgAA
AT1SAAABO9wAAAJ04AAAA8NaAAADgeoAAABPeQAAAi2gAAAAFjsAAAGgnAAABHVKAAABpsgAAAMMHgAAALpUAAABV+gAAAMoAgAA
An7KAAAAhPIAAASLVgAAAGDaAAAAPSwAAAOddAAABB4yAAACBpcAAAFbPxUAFaIBFagBLBW0ARUQFQYVBhwYBQAABroSGAUAAAAA
ABYaKAUAAAa6EhgFAAAAAAAREQAAAFHwUA8AAAAL6u/9798WAQXi7ywBAgAGFUAgDAABAEBhHEiiLEzjAA8AAZEETBRQWReAZdrG
AR0AeB8YiiNAAqV5AmiqrmzrAi8cyzMAAABd2zcAABUEGVw1ABgGc2NoZW1hFQgAFQQlAhgIb3JkZXJfaWQAFQwlAhgHcHJvZHVj
dCUATBwAAAAVDhUKFQIYDGdyb3NzX2Ftb3VudCUKFQQVFCxcFQQVFAAAABUOFQoVAhgPZGlzY291bnRfYW1vdW50JQoVBBUULFwV
BBUUAAAAFrQBGRwZTCYAHBUEGTUABhAZGAhvcmRlcl9pZBUCFrQBFoAOFtQIJpgGJggcGAgKrgEAAAAAABgIsa0BAAAAAAAWACgI
Cq4BAAAAAAAYCLGtAQAAAAAAEREAGSwVBBUAFQIAFQAVEBUCADwpBhkmALQBAAAAJgAcFQwZNQAGEBkYB3Byb2R1Y3QVAha0ARao
AxauAybICibcCBw2ACgOV2lyZWxlc3MgTW91c2UYCERlc2sgTWF0EREAGSwVBBUAFQIAFQAVEBUCADwWvg4ZBhkmALQBAAAAJgAc
FQ4ZNQAGEBkYDGdyb3NzX2Ftb3VudBUCFrQBFswJFtwJJroTJooMHBgFAAAWwpAYBQAAANLwFgAoBQAAFsKQGAUAAADS8BERABks
FQQVABUCABUAFRAVAgA8KQYZJgC0AQAAACYAHBUOGTUABhAZGA9kaXNjb3VudF9hbW91bnQVAha0ARbeBhboBia6GibmFRwYBQAA
BroSGAUAAAAAABYaKAUAAAa6EhgFAAAAAAAREQAZLBUEFQAVAgAVABUQFQIAPCkGGSYamgEAAAAW0iEWtAEmCBbGHAAZHBgMQVJS
T1c6c2NoZW1hGKADLy8vLy96QUJBQUFRQUFBQUFBQUtBQXdBQmdBRkFBZ0FDZ0FBQUFBQkJBQU1BQUFBQ0FBSUFBQUFCQUFJQUFB
QUJBQUFBQVFBQUFERUFBQUFoQUFBQUVBQUFBQUVBQUFBWFAvLy93QUFBUWNRQUFBQUlBQUFBQVFBQUFBQUFBQUFEd0FBQUdScGMy
TnZkVzUwWDJGdGIzVnVkQURJLy8vL0NnQUFBQUlBQUFDVS8vLy9BQUFCQnhBQUFBQW9BQUFBQkFBQUFBQUFBQUFNQUFBQVozSnZj
M05mWVcxdmRXNTBBQUFBQUFnQURBQUVBQWdBQ0FBQUFBb0FBQUFDQUFBQTFQLy8vd0FBQVFVUUFBQUFIQUFBQUFRQUFBQUFBQUFB
QndBQUFIQnliMlIxWTNRQUJBQUVBQVFBQUFBUUFCUUFDQUFHQUFjQURBQUFBQkFBRUFBQUFBQUFBUUlRQUFBQUpBQUFBQVFBQUFB
QUFBQUFDQUFBQUc5eVpHVnlYMmxrQUFBQUFBZ0FEQUFJQUFjQUNBQUFBQUFBQUFGQUFBQUEAGCBwYXJxdWV0LWNwcC1hcnJvdyB2
ZXJzaW9uIDI1LjAuMRlMHAAAHAAAHAAAHAAAABcEAABQQVIx
""",
    "sales_south": """
UEFSMRUEFYAKFZwFTBWgARUAEgAAgAUMYVsDAAEBAGINCABjDQgAZA0IAGUNCABmDQgAZw0IAGgNCABpDQgAag0IAGsNCABsDQgA
bQ0IAG4NCABvDQgAcA0IAHENCAByDQgAcw0IAHQNCAB1DQgAdg0IAHcNCAB4DQgAeQ0IAHoNCAB7DQgAfA0IAH0NCAB+DQgAfw0I
AIANCACBDQgAgg0IAIMNCACEDQgAhQ0IAIYNCACHDQgAiA0IAIkNCACKDQgAiw0IAIwNCACNDQgAjg0IAI8NCACQDQgAkQ0IAJIN
CACTDQgAlA0IAJUNCACWDQgAlw0IAJgNCACZDQgAmg0IAJsNCACcDQgAnQ0IAJ4NCACfDQgAoA0IAKENCACiDQgAow0IAKQNCACl
DQgApg0IAKcNCACoDQgAqQ0IAKoNCACrDQgArA0IAK0NCACuDQg8r1sDAAAAAACwWwMAAAAAABUAFZ4BFaQBLBWgARUQFQYVBhwY
CLBbAwAAAAAAGAhhWwMAAAAAABYAKAiwWwMAAAAAABgIYVsDAAAAAAAREQAAAE/wTgMAAACgAQEHFYCAYEAoGA6IhGLBaDgekIhk
QqlYLpiMZsPpeD6gkGhEKplOqJRqxWq5XrCYbEar2W64nG7H6/l+wKBwSCwaj8ikcslsOp8VBBXAARXGAUwVDhUAEgAAYPBfCwAA
AE1vbml0b3IgQXJtCQAAAFVTQi1DIEh1YggAAABGb290cmVzdAwAAABMYXB0b3AgU3RhbmQGAAAAV2ViY2FtCAAAAERlc2sgTWF0
DgAAAFdpcmVsZXNzIE1vdXNlFQAVThVSLBWgARUQFQYVBhw2ACgOV2lyZWxlc3MgTW91c2UYCERlc2sgTWF0EREAAAAnmAMAAACg
AQEDFYg21unmgN3IYktZRh4juAag2R2pmQULljSBck1GVBUEFf4DFYYETBVmFQASAAD/AfA+AAACvE0AAAAhogAAAvJxAAAFOVMA
AABCzAAAAkHLAAAA1ZMAAAHYRAAAAk6mAAAAKIUAAAB69wAAAEODAAAAAQLwuwL7YgAAAQgaAAAFbfwAAAChFwAAA2zqAAAAZ24A
AAIe5wAAAaNqAAACEgQAAACrUAAAAXvYAAABrGoAAAYBJgAAAqWUAAAAWtgAAAIm4wAAAZHkAAABE80AAAI8+AAAAb4WAAABHAYA
AAAUCgAABcXgAAADBWAAAABUqwAAAKoQAAACEg4AAAE0mgAAAP5MAAABUDIAAAJqwwAABhbyAAAAqGIAAAKOyQAAAoKQAAADk6cA
AAL10AAAAOdcFQAVmgEVoAEsFaABFRAVBhUGHBgFAAAGFvIYBQAAAAAAFhgoBQAABhbyGAUAAAAAABERAAAATfBMEQAAAAP2FAEF
/KsQAQf+/X8QAQMeBhNAIAxEYRxIoixM4zwMFEkTQ1UWc2EMwzCZtjEMw3Cd9zFgKI5kaZ4okzIqs7Kt+zJwLDMAAAAVBBWgBhWq
BkwVoAEVABIAAJAD9I8BAAAB4mwAAA0GxAAAAeB4AAAFGdwAABFV5AAAFrZ0AAABcxgAAA8L9AAABY/UAAAHGGgAAAf0uAAAALAs
AAACSYwAAAPEdAAACbGkAAAVTLwAABSiCAAAFOIYAAAWaLgAABay8AAABFdUAAAFuHQAABBPyAAABrvUAAAQUCwAAACLdAAAEGIk
AAALgIgAAACyhAAABC60AAAORLwAAAGAYAAAFfkAAAAFtPAAAAlMFAAAEzFIAAAWEgAAABS5eAAAARNkAAAUA9QAABYOfAAABG+M
AAAWQnAAAAYEPAAACTJMAAAWPPgAABD0oAAAB2ucAAAJPBAAABWMBAAAFmGwAAAHF6AAAAtRqAAACEw4AAAKFggAAAGQyAAAFJ4g
AAAS4ZgAAAI0dAAACE3IAAAH9qwAAAicTAAAA6NAAAAJ7vgAAA6u/AAAFUxYAAAFtbgAABDSpAAAC4J8AAAUTHwAAASyvAAAEQ08
AAAU6rAAAAg3IAAABCPEAAAMVWwAAAvXQAAABVDwAAAKcjgAABGllBUAFZ4BFaQBLBWgARUQFQYVBhwYBQAAFrZ0GAUAAACLdBYA
KAUAABa2dBgFAAAAi3QREQAAAE/wTgMAAACgAQEHFYCAYEAoGA6IhGLBaDgekIhkQqlYLpiMZsPpeD6gkGhEKplOqJRqxWq5XrCY
bEar2W64nG7H6/l+wKBwSCwaj8ikcslsOp8VBBlcNQAYBnNjaGVtYRUIABUEJQIYCG9yZGVyX2lkABUMJQIYB3Byb2R1Y3QlAEwc
AAAAFQ4VChUCGA9kaXNjb3VudF9hbW91bnQlChUEFRQsXBUEFRQAAAAVDhUKFQIYDGdyb3NzX2Ftb3VudCUKFQQVFCxcFQQVFAAA
ABagARkcGUwmABwVBBk1AAYQGRgIb3JkZXJfaWQVAhagARbEDBbmBybGBSYIHBgIsFsDAAAAAAAYCGFbAwAAAAAAFgAoCLBbAwAA
AAAAGAhhWwMAAAAAABERABksFQQVABUCABUAFRAVAgA8KQYZJgCgAQAAACYAHBUMGTUABhAZGAdwcm9kdWN0FQIWoAEWkgMWnAMm
1Akm7gccNgAoDldpcmVsZXNzIE1vdXNlGAhEZXNrIE1hdBERABksFQQVABUCABUAFRAVAgA8FvALGQYZJgCgAQAAACYAHBUOGTUA
BhAZGA9kaXNjb3VudF9hbW91bnQVAhagARakBhayBiawDyaKCxwYBQAABhbyGAUAAAAAABYYKAUAAAYW8hgFAAAAAAAREQAZLBUE
FQAVAgAVABUQFQIAPCkGGSYYiAEAAAAmABwVDhk1AAYQGRgMZ3Jvc3NfYW1vdW50FQIWoAEWzAgW3AgmiBgmvBEcGAUAABa2dBgF
AAAAi3QWACgFAAAWtnQYBQAAAIt0EREAGSwVBBUAFQIAFQAVEBUCADwpBhkmAKABAAAAFsYeFqABJggWkBoAGRwYDEFSUk9XOnNj
aGVtYRigAy8vLy8vekFCQUFBUUFBQUFBQUFLQUF3QUJnQUZBQWdBQ2dBQUFBQUJCQUFNQUFBQUNBQUlBQUFBQkFBSUFBQUFCQUFB
QUFRQUFBREVBQUFBaEFBQUFFQUFBQUFFQUFBQVhQLy8vd0FBQVFjUUFBQUFJQUFBQUFRQUFBQUFBQUFBREFBQUFHZHliM056WDJG
dGIzVnVkQUFBQUFESS8vLy9DZ0FBQUFJQUFBQ1UvLy8vQUFBQkJ4QUFBQUFvQUFBQUJBQUFBQUFBQUFBUEFBQUFaR2x6WTI5MWJu
UmZZVzF2ZFc1MEFBZ0FEQUFFQUFnQUNBQUFBQW9BQUFBQ0FBQUExUC8vL3dBQUFRVVFBQUFBSEFBQUFBUUFBQUFBQUFBQUJ3QUFB
SEJ5YjJSMVkzUUFCQUFFQUFRQUFBQVFBQlFBQ0FBR0FBY0FEQUFBQUJBQUVBQUFBQUFBQVFJUUFBQUFKQUFBQUFRQUFBQUFBQUFB
Q0FBQUFHOXlaR1Z5WDJsa0FBQUFBQWdBREFBSUFBY0FDQUFBQUFBQUFBRkFBQUFBABggcGFycXVldC1jcHAtYXJyb3cgdmVyc2lv
biAyNS4wLjEZTBwAABwAABwAABwAAAAXBAAAUEFSMQ==
""",
}

_data_dir = pathlib.Path("data") / "008-union-swapped-columns"
_data_dir.mkdir(parents=True, exist_ok=True)
for _name, _encoded in _DATA.items():
    (_data_dir / f"{_name}.parquet").write_bytes(base64.b64decode(_encoded))

sales_north = spark.read.parquet((_data_dir / "sales_north.parquet").resolve().as_posix())
sales_north.createOrReplaceTempView("sales_north")
sales_south = spark.read.parquet((_data_dir / "sales_south.parquet").resolve().as_posix())
sales_south.createOrReplaceTempView("sales_south")

for _name in _DATA:
    print(f"{_name}: {spark.table(_name).count()} rows")

In [ ]:
#@title Grader (run this cell, no need to read it)
"""Result grading shared by the build script and every generated notebook.

build_problem.py pastes this file into the notebook verbatim, so it must stay
standalone: standard library only, no imports from the rest of the toolkit.

A result is reduced to a "summary": its columns and types, its row count and
SHA-256 fingerprints of its normalized rows. Only the summary of the correct
answer is shipped in a notebook, never the rows themselves.
"""
import datetime
import decimal
import hashlib
import json
import math

FLOAT_DECIMALS = 6


def normalize_value(value):
    """Turn one cell value into something json.dumps renders the same way everywhere."""
    if value is None or isinstance(value, (bool, int, str)):
        return value
    if isinstance(value, float):
        if math.isnan(value):
            return "NaN"
        if math.isinf(value):
            return "Infinity" if value > 0 else "-Infinity"
        return round(value, FLOAT_DECIMALS) + 0.0  # + 0.0 turns -0.0 into 0.0
    if isinstance(value, decimal.Decimal):
        return format(value, "f")
    if isinstance(value, (datetime.datetime, datetime.date)):
        return value.isoformat()
    if isinstance(value, (bytes, bytearray)):
        return bytes(value).hex()
    if hasattr(value, "asDict"):  # pyspark Row (struct column)
        return {key: normalize_value(item) for key, item in value.asDict().items()}
    if isinstance(value, dict):  # map column
        pairs = [[normalize_value(key), normalize_value(item)] for key, item in value.items()]
        return sorted(pairs, key=_encode)
    if isinstance(value, (list, tuple)):
        return [normalize_value(item) for item in value]
    return str(value)


def _encode(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"))


def _digest(lines):
    sha = hashlib.sha256()
    for line in lines:
        sha.update(line.encode("utf-8"))
        sha.update(b"\n")
    return sha.hexdigest()


def describe_schema(df):
    """Column names and Spark types, sorted by name so column order is ignored."""
    return sorted([field.name, field.dataType.simpleString()] for field in df.schema.fields)


def summarize_rows(columns, rows, order_matters=False):
    """columns: sorted [name, type] pairs. rows: normalized values in that column order."""
    encoded = [_encode(row) for row in rows]
    if not order_matters:
        encoded.sort()
    column_fingerprints = {}
    for index, (name, _) in enumerate(columns):
        column_fingerprints[name] = _digest(sorted(_encode(row[index]) for row in rows))
    return {
        "columns": [list(column) for column in columns],
        "row_count": len(rows),
        "order_matters": order_matters,
        "fingerprint": _digest([_encode(columns)] + encoded),
        "column_fingerprints": column_fingerprints,
    }


def summarize(df, order_matters=False):
    columns = describe_schema(df)
    names = [name for name, _ in columns]
    # collect() returns `timestamp` values in the machine's local time zone, which differs
    # between the build machine and Colab. Casting to string inside Spark uses the session
    # time zone instead (UTC in both places). `timestamp_ntz` columns are not affected.
    expressions = [
        "CAST(`%s` AS STRING) AS `%s`" % (name, name) if kind == "timestamp" else "`%s`" % name
        for name, kind in columns
    ]
    collected = df.selectExpr(*expressions).collect()
    rows = [[normalize_value(row[name]) for name in names] for row in collected]
    return summarize_rows(columns, rows, order_matters)


def grade(df, expected):
    """Compare a DataFrame to the expected summary. Returns (passed, hints).

    The hints say what kind of thing is wrong (columns, types, row count, which
    column) but never what the right values are.
    """
    if not hasattr(df, "schema") or not hasattr(df, "collect"):
        return False, ["check() needs a Spark DataFrame, got %s." % type(df).__name__]

    names = [field.name for field in df.schema.fields]
    duplicates = sorted({name for name in names if names.count(name) > 1})
    if duplicates:
        return False, ["Your result has duplicate column names: %s." % ", ".join(duplicates)]

    actual = dict(describe_schema(df))
    wanted = dict(expected["columns"])
    hints = []
    missing = sorted(set(wanted) - set(actual))
    extra = sorted(set(actual) - set(wanted))
    if missing:
        hints.append("Missing columns: %s." % ", ".join(missing))
    if extra:
        hints.append("Unexpected columns: %s." % ", ".join(extra))
    for name in sorted(set(wanted) & set(actual)):
        if wanted[name] != actual[name]:
            hints.append("Column %s has type %s, expected %s." % (name, actual[name], wanted[name]))
    if hints:
        return False, hints

    row_count = df.count()
    if row_count != expected["row_count"]:
        return False, [
            "Row count is wrong: your result has %d rows, expected %d."
            % (row_count, expected["row_count"])
        ]

    summary = summarize(df, expected["order_matters"])
    if summary["fingerprint"] == expected["fingerprint"]:
        return True, []

    wrong = sorted(
        name
        for name, fingerprint in summary["column_fingerprints"].items()
        if fingerprint != expected["column_fingerprints"][name]
    )
    if wrong:
        return False, ["Columns and row count are right, but values are wrong in: %s." % ", ".join(wrong)]
    if expected["order_matters"]:
        return False, ["The values are right, but the rows are in the wrong order."]
    return False, ["Each column has the right set of values, but they are combined into the wrong rows."]


def make_check(expected):
    def check(df):
        passed, hints = grade(df, expected)
        if passed:
            print("Correct. Your result matches the expected output.")
        else:
            print("Not correct yet.")
            for hint in hints:
                print("  - " + hint)
        # Returns nothing, so a notebook cell ending in check(result) prints only the message.

    return check

EXPECTED = {'column_fingerprints': {'net_revenue': 'd967fef0457d89d953730ed8f9a721ad846b0fcabf3d40a2bbcaf0089beb013a',
                         'orders': 'b41cbf4419a209f44102cefb5cea0e29240bf5ffa525c6174fb8f9453a173cab',
                         'product': '326749fe1310549e5d3579071400b4f013ac91d36f9fe745ffb329b62e333f71'},
 'columns': [['net_revenue', 'decimal(12,2)'], ['orders', 'bigint'], ['product', 'string']],
 'fingerprint': 'bce3ad5adbbba3c4504ba3ade6a7970dc0bd60301c500ebdf3c2b1cebdf01de2',
 'order_matters': False,
 'row_count': 8}

check = make_check(EXPECTED)

## The situation

Two regional teams each export their sales. An AI assistant wrote the job that
combines the two exports and reports net revenue per product. It runs without
errors and the order counts match what the teams expect.

The net revenue does not. Some products look far less profitable than they are,
and two products have negative revenue.

The AI-written code is in the answer cell. Find the bug and fix it.

## Input tables

### `sales_north`

One row per order from the north region.

| Column | Type | Description |
|---|---|---|
| `order_id` | BIGINT | Unique across both regions. |
| `product` | STRING | Product name. |
| `gross_amount` | DECIMAL(10,2) | Order value before discount. |
| `discount_amount` | DECIMAL(10,2) | Discount given. NULL means no discount was recorded. |

### `sales_south`

One row per order from the south region. Same columns as `sales_north`. Run
`sales_south.printSchema()` before you trust that.

## Requirement

Return net revenue per product across both regions.

- The net amount of an order is `gross_amount` minus `discount_amount`.
  A NULL discount counts as 0.
- `orders` is the number of orders of the product in both regions together.
- `net_revenue` is the sum of the net amounts of those orders.
- A product sold in only one region still appears.

## Expected output

One row per product.

| Column | Type | Description |
|---|---|---|
| `product` | STRING | |
| `orders` | BIGINT | Number of orders. |
| `net_revenue` | DECIMAL(12,2) | Sum of gross minus discount. |

In [ ]:
sales_north.show(5)
sales_south.show(5)

## Your answer

The cell below is the AI-written code. It runs, but its output is wrong. Find the bug and fix it. The tables are available as DataFrames and as views: `sales_north`, `sales_south`.

In [ ]:
from pyspark.sql import functions as F


def solve(spark):
    """Net revenue per product across both regions."""
    north = spark.table("sales_north")
    south = spark.table("sales_south")

    # Combine both regions into a single DataFrame
    all_sales = north.union(south)

    # Net amount per order, treating a missing discount as zero
    all_sales = all_sales.withColumn(
        "net_amount",
        F.col("gross_amount") - F.coalesce(F.col("discount_amount"), F.lit(0)),
    )

    # Aggregate per product
    result = all_sales.groupBy("product").agg(
        F.count("*").alias("orders"),
        F.sum("net_amount").cast("decimal(12,2)").alias("net_revenue"),
    )

    return result


result = solve(spark)
result.show()

In [ ]:
check(result)